# ohie_qje_test_one

> 对应代码：`EconometricsCode/code_in_notes/Chap.9/ohie_qje_test_one.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

加载俄勒冈健康保险实验（OHIE）数据，该数据来自 QJE 发表的随机实验研究。

In [ ]:
%%stata
clear
set more off
use ../datasets/OHIE_QJE.dta

把需要比较的基线特征变量存入局部宏 ys：出生年份、性别、英语能力、是否本人填表、是否首日申请、是否有电话。这些都是随机分组前可观测的个体特征。

In [ ]:
%%stata
// 需要进行比较的特征
local ys "birthyear_list female_list english_list self_list first_day_list have_phone_list"

用局部宏 repapp 控制 outreg2 的输出方式：第一次写表用 replace 覆盖，之后用 append 追加，从而把多个回归结果写入同一张表。

In [ ]:
%%stata
// 挨个比较
local repapp "replace"

循环对每个基线特征做平衡性检验（balance test）：以特征变量为被解释变量，对处理虚拟变量 treatment 做加权最小二乘回归（[iw=weight_12m] 为 12 个月随访的抽样权重），使用稳健标准误。若随机化成功，treatment 的系数应在统计上不显著。回归后 outreg2 输出结果，并把 repapp 切换为 append 使后续结果追加到同一文件。

In [ ]:
%%stata
foreach v of varlist `ys'{
	reg `v' treatment [iw=weight_12m ], robust
	outreg2 using ohie_qje_test_one.tex, `repapp'
	local repapp "append"
	test treatment
}